# Bronze Layer — Managed Delta Tables with Auto Loader Ingestion

## Design Decisions
- **Managed Delta tables** (not external) so Unity Catalog owns both metadata and data lifecycle.
- **Auto Loader (`cloudFiles`)** for incremental, scalable file ingestion — only new files are processed on each run.
- **Explicit schemas** provided to Auto Loader — avoids `CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE` when the source path is empty at stream startup.
- **NOT NULL constraints** applied at the Bronze layer as the first data quality gate.
- **Partitioned by `ingestion_date`** for query pruning and efficient time-travel.

> Prerequisite: Run `0.config.ipynb` first (or let the Workflow chain handle it via `%run`).

In [0]:
# ── Step 0: Load shared configuration ────────────────────────────────────────
%run ./0.config

In [0]:
# ── Step 1: Ensure catalog / schema exist ────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA} MANAGED LOCATION '{BRONZE_PATH}'")

In [ ]:
# ── Step 2: Create target managed Delta tables (DDL with NOT NULL constraints) ─

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'drivers')} (
    driverId       INT       NOT NULL,
    driverRef      STRING    NOT NULL,
    number         INT,
    code           STRING,
    name           STRUCT<forename: STRING, surname: STRING>,
    dob            STRING,
    nationality    STRING,
    url            STRING,
    ingestion_date DATE      NOT NULL,
    source_file    STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'results')} (
    resultId        INT     NOT NULL,
    raceId          INT     NOT NULL,
    driverId        INT     NOT NULL,
    constructorId   INT     NOT NULL,
    number          INT,
    grid            INT,
    position        INT,
    positionText    STRING,
    positionOrder   INT,
    points          DOUBLE,
    laps            INT,
    time            STRING,
    milliseconds    INT,
    fastestLap      INT,
    rank            INT,
    fastestLapTime  STRING,
    fastestLapSpeed STRING,
    statusId        INT,
    ingestion_date  DATE    NOT NULL,
    source_file     STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

In [ ]:
# ── Step 3: Auto Loader — incremental ingestion of drivers.json ───────────────
# Explicit schema avoids CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE when source path is
# empty at stream start. cloudFiles.inferColumnTypes is intentionally NOT used.
# _metadata.file_path is used instead of input_file_name() (not supported in UC).

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType
)

drivers_schema = StructType([
    StructField("driverId",    IntegerType(), nullable=True),
    StructField("driverRef",   StringType(),  nullable=True),
    StructField("number",      IntegerType(), nullable=True),
    StructField("code",        StringType(),  nullable=True),
    StructField("name",        StructType([
        StructField("forename", StringType(), nullable=True),
        StructField("surname",  StringType(), nullable=True),
    ]),                        nullable=True),
    StructField("dob",         StringType(),  nullable=True),
    StructField("nationality", StringType(),  nullable=True),
    StructField("url",         StringType(),  nullable=True),
])

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{BRONZE_PATH}_schema/drivers")
    .option("multiLine", "true")
    .schema(drivers_schema)                              # explicit schema — no scan at startup
    .load(f"{BRONZE_PATH}drivers.json")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.col("_metadata.file_path"))  # UC-compatible alternative
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_PATH}_checkpoint/drivers")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)                          # process all backlog then stop
    .toTable(fq(BRONZE_SCHEMA, "drivers"))
    .awaitTermination()
)

In [ ]:
# ── Step 4: Auto Loader — incremental ingestion of results.json ───────────────

from pyspark.sql.types import DoubleType

results_schema = StructType([
    StructField("resultId",        IntegerType(), nullable=True),
    StructField("raceId",          IntegerType(), nullable=True),
    StructField("driverId",        IntegerType(), nullable=True),
    StructField("constructorId",   IntegerType(), nullable=True),
    StructField("number",          IntegerType(), nullable=True),
    StructField("grid",            IntegerType(), nullable=True),
    StructField("position",        IntegerType(), nullable=True),
    StructField("positionText",    StringType(),  nullable=True),
    StructField("positionOrder",   IntegerType(), nullable=True),
    StructField("points",          DoubleType(),  nullable=True),
    StructField("laps",            IntegerType(), nullable=True),
    StructField("time",            StringType(),  nullable=True),
    StructField("milliseconds",    IntegerType(), nullable=True),
    StructField("fastestLap",      IntegerType(), nullable=True),
    StructField("rank",            IntegerType(), nullable=True),
    StructField("fastestLapTime",  StringType(),  nullable=True),
    StructField("fastestLapSpeed", StringType(),  nullable=True),
    StructField("statusId",        IntegerType(), nullable=True),
])

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{BRONZE_PATH}_schema/results")
    .option("multiLine", "true")
    .schema(results_schema)                              # explicit schema — no scan at startup
    .load(f"{BRONZE_PATH}results.json")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.col("_metadata.file_path"))  # UC-compatible alternative
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_PATH}_checkpoint/results")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(fq(BRONZE_SCHEMA, "results"))
    .awaitTermination()
)

In [ ]:
# ── Step 5: Data Quality Validation — row counts & NOT NULL checks ────────────

def validate_bronze(schema: str, table: str, pk_col: str):
    full_name = fq(schema, table)
    df = spark.table(full_name)
    total      = df.count()
    null_pk    = df.filter(F.col(pk_col).isNull()).count()
    null_date  = df.filter(F.col("ingestion_date").isNull()).count()

    assert total   > 0,  f"[DQ FAIL] {full_name}: table is empty!"
    assert null_pk == 0, f"[DQ FAIL] {full_name}: {null_pk} NULL values in primary key '{pk_col}'"
    assert null_date == 0, f"[DQ FAIL] {full_name}: {null_date} NULL values in ingestion_date"

    print(f"[DQ PASS] {full_name}: {total:,} rows | 0 NULL PKs | 0 NULL dates")

validate_bronze(BRONZE_SCHEMA, "drivers", "driverId")
validate_bronze(BRONZE_SCHEMA, "results", "resultId")